#### 1. Classes com especificações para geração de dados randômicos (metadata, transformers e debuggers)

Classes especificadas:
- FakeCustomer();
- FakeOrder()

In [0]:
import faker
from rand_engine.core.distinct_core import DistinctCore
from rand_engine.core.numeric_core import NumericCore
from rand_engine.core.datetime_core import DatetimeCore
from rand_engine.core.distinct_utils import DistinctUtils
from pandas import DataFrame as PandasDF
import numpy as np
import pandas as pd
from datetime import datetime as dt, timedelta

class FakeCustomer:

    def __init__(self):
        self.faker = faker.Faker(locale="pt_BR")

    def metadata(self):
        return {
        "user_id": {
            "method": NumericCore.gen_ints_zfilled,
            "parms": dict(length=14)
        },
        "user_type": {     
            "method": DistinctCore.gen_distincts_untyped,
            "parms": dict(distinct=DistinctUtils.handle_distincts_lvl_1({"standard": 80,"premium":15, "gold": 5, None: 7}, 1))
        },
        "first_name": {
            "method": DistinctCore.gen_distincts_typed,
            "parms": dict(distinct=[self.faker.first_name() for _ in range(1000)])
        },
        "last_name": {
            "method": DistinctCore.gen_distincts_typed,
            "parms": dict(distinct=[f"{self.faker.last_name()} {self.faker.last_name()}" for _ in range(10000)])
        },
        "income": {
            "method": NumericCore.gen_floats_normal,
            "parms": dict(mean=10000, std=3000, round=2)
        },
        "balance": {
            "method": NumericCore.gen_floats_normal,
            "parms": dict(mean=5000, std=3000, round=2)
        },
        "profession": {
            "method": DistinctCore.gen_distincts_typed,
            "parms": dict(distinct=[self.faker.job() for _ in range(100)])
        },
        "birth_date": dict(
            method=DatetimeCore.gen_datetimes, 
            parms=dict(start='1971-07-05', end='2013-07-06', format_in="%Y-%m-%d", format_out="%d/%m/%Y")
        ),
        "signup_date": dict(
            method=DatetimeCore.gen_timestamps,
            parms=dict(start="01-01-2021", end="31-12-2025", format="%d-%m-%Y")
        )
    }

    def transformer(self, **kwargs):
        def wrapped_transformer(df: PandasDF) -> PandasDF:
            df["income"] = np.where(df["income"] < 0, 0, df["income"])
            for k, v in kwargs.items(): df[k] = v
            for col in df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns:
                df[col] = df[col].dt.strftime('%Y-%m-%dT%H:%M:%S')
            return df
        return wrapped_transformer

    def transformer_cdc_update(self, null_rate, **kwargs):
        def wrapped_transformer(df: PandasDF) -> PandasDF:
            for col in df.columns:
                df[col] = np.where(np.random.random(df.shape[0]) < null_rate, None, df[col])
            for k, v in kwargs.items(): df[k] = v
            for col in df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns:
                df[col] = df[col].dt.strftime('%Y-%m-%dT%H:%M:%S')
            return df
        return wrapped_transformer
    
    def debugger(self):
        
        
        data = {
            "id": [1, 2, 3],
            "name": ["marco", "gisele", "tauan"],
            "age": [34, 34, 2]
        }
        df = pd.DataFrame(data)
        return df
    

class FakeOrders:

    def __init__(self):
        self.faker = faker.Faker(locale="pt_BR")

    def metadata(self):
        return {
            "order_id": {
                "method": NumericCore.gen_ints_zfilled,
                "parms": dict(length=16)
            },
            "user_id": {
                "method": NumericCore.gen_ints_zfilled,
                "parms": dict(length=10)
            },
            "product_id": {
                "method": NumericCore.gen_ints_zfilled,
                "parms": dict(length=3)
            },
            "user_type": {     
            "method": DistinctCore.gen_distincts_untyped,
            "parms": dict(distinct=DistinctUtils.handle_distincts_lvl_1({"standard": 80,"premium":15, "gold": 5, None: 7}, 1))
            },
            "device": {
                "method": DistinctCore.gen_distincts_typed,
                "parms": dict(distinct=["IOS", "Android", "Desktop"])
            },
            "traffic_source": {
                "method": DistinctCore.gen_distincts_typed,
                "parms": dict(distinct=["website", "linkedin", "email"])
            }
        }

    def transformer(self):
        def wrapped_transformer(df: PandasDF) -> PandasDF:
            for col in df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns:
                df[col] = df[col].dt.strftime('%Y-%m-%dT%H:%M:%S')
            start_time = dt.now() - timedelta(minutes=10)
            end_time = dt.now()
            random_times = pd.to_datetime(np.random.uniform(start_time.timestamp(), end_time.timestamp(), size=len(df)), unit='s').strftime('%Y-%m-%dT%H:%M:%S')
            df["created_at"] = random_times
            return df
        return wrapped_transformer
    

    def debugger(self):
        data = {
            "id": [1, 2, 3],
            "name": ["marco", "gisele", "tauan"],
            "age": [34, 34, 2]
        }
        df = pd.DataFrame(data)
        return df


## CDC Generator

